In [ ]:
# Install and import necessary libraries
!pip install sentencepiece
!pip install transformers datasets rouge-score
import torch
from transformers import PegasusForConditionalGeneration, PegasusTokenizer
from datasets import load_dataset, load_metric

# Load the dataset
dataset = load_dataset("cnn_dailymail", "3.0.0")

# Load the Pegasus model and tokenizer
model_name = "google/pegasus-cnn_dailymail"
tokenizer = PegasusTokenizer.from_pretrained(model_name)
model = PegasusForConditionalGeneration.from_pretrained(model_name)

# Function to generate summaries
def generate_summary(article):
    inputs = tokenizer(article, return_tensors="pt", padding="max_length", truncation=True, max_length=512)
    summary_ids = model.generate(inputs["input_ids"], num_beams=4, max_length=150, early_stopping=True)
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

# Generate summaries for the test set
test_samples = dataset["test"].select(range(10))  # Use a small subset for quick results
predicted_summaries = [generate_summary(sample["article"]) for sample in test_samples]

# Print the first predicted summary
print("Original Article:", test_samples[0]["article"])
print("Predicted Summary:", predicted_summaries[0])


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.5/510.5 kB 6.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 19.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 15.3 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24933 sha256=f0f6dd0257345e63cfa7c32958633a6b5d9ae566f1f368ad935a6b49817d0499
  Stored in directory: /root/.cache/pip/wheels/5f/dd/89/461065a73be61a532ff8599a28e9beef17985c9e9c31e541b4
Successfully built rouge-score


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

...

config.json:   0%|          | 0.00/1.12k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-cnn_dailymail and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


generation_config.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

Original Article: (CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territories. The formal accession was marked with a ceremony at The Hague, in the Netherlands, where the court is based. The Palestinians signed the ICC's founding Rome Statute in January, when they also accepted its jurisdiction over alleged crimes committed "in the occupied Palestinian territory, including East Jerusalem, since June 13, 2014." Later that month, the ICC opened a preliminary examination into the situation in Palestinian territories, paving the way for possible war crimes investigations against Israelis. As members of the court, Palestinians may be subject to counter-charges as well. Israel and the United States, neither of which is an ICC member, opposed the Palestinians' efforts to join the body. But Palestinian Foreign Minister Riad al-Malki, speaking at Wednes

In [ ]:
from rouge_score import rouge_scorer

# Initialize the Rouge scorer
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

# Function to calculate Rouge scores for a list of generated summaries and reference summaries
def calculate_rouge_scores(generated_summaries, reference_summaries):
    scores = {'rouge1': [], 'rouge2': [], 'rougeL': []}
    for gen_summary, ref_summary in zip(generated_summaries, reference_summaries):
        score = scorer.score(ref_summary, gen_summary)
        scores['rouge1'].append(score['rouge1'].fmeasure)
        scores['rouge2'].append(score['rouge2'].fmeasure)
        scores['rougeL'].append(score['rougeL'].fmeasure)
    # Calculate the average scores
    avg_scores = {key: sum(values) / len(values) for key, values in scores.items()}
    return avg_scores

# Reference summaries
reference_summaries = [sample["highlights"] for sample in test_samples]

# Calculate Rouge scores for Pegasus
pegasus_rouge_scores = calculate_rouge_scores(predicted_summaries, reference_summaries)
print("Pegasus Rouge Scores:", pegasus_rouge_scores)


Pegasus Rouge Scores: {'rouge1': 0.3631632637685512, 'rouge2': 0.14700211444333067, 'rougeL': 0.2553976311936206}


In [ ]:
# Install and import necessary libraries
!pip install transformers datasets rouge-score
import torch
from transformers import T5ForConditionalGeneration, T5Tokenizer
from datasets import load_dataset, load_metric

# Load the dataset
dataset = load_dataset("cnn_dailymail", "3.0.0")

# Load the T5 model and tokenizer
model_name = "t5-small"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

# Function to generate summaries
def generate_summary(article):
    inputs = tokenizer("summarize: " + article, return_tensors="pt", padding="max_length", truncation=True, max_length=512)
    summary_ids = model.generate(inputs["input_ids"], num_beams=4, max_length=150, early_stopping=True)
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

# Generate summaries for the test set
test_samples = dataset["test"].select(range(10))  # Use a small subset for quick results
predicted_summaries = [generate_summary(sample["article"]) for sample in test_samples]

# Print the first predicted summary
print("Original Article:", test_samples[0]["article"])
print("Predicted Summary:", predicted_summaries[0])


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Original Article: (CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territories. The formal accession was marked with a ceremony at The Hague, in the Netherlands, where the court is based. The Palestinians signed the ICC's founding Rome Statute in January, when they also accepted its jurisdiction over alleged crimes committed "in the occupied Palestinian territory, including East Jerusalem, since June 13, 2014." Later that month, the ICC opened a preliminary examination into the situation in Palestinian territories, paving the way for possible war crimes investigations against Israelis. As members of the court, Palestinians may be subject to counter-charges as well. Israel and the United States, neither of which is an ICC member, opposed the Palestinians' efforts to join the body. But Palestinian Foreign Minister Riad al-Malki, speaking at Wednes

In [ ]:
# Reference summaries
reference_summaries = [sample["highlights"] for sample in test_samples]

# Calculate Rouge scores for T5
t5_rouge_scores = calculate_rouge_scores(predicted_summaries, reference_summaries)
print("T5 Rouge Scores:", t5_rouge_scores)


T5 Rouge Scores: {'rouge1': 0.3631632637685512, 'rouge2': 0.14700211444333067, 'rougeL': 0.2553976311936206}


In [ ]:
# Install and import necessary libraries
!pip install transformers datasets
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from datasets import load_dataset

# Load the dataset
dataset = load_dataset("cnn_dailymail", "3.0.0")

# Load the BERT model and tokenizer
model_name = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForSequenceClassification.from_pretrained(model_name, num_labels=2)  # Binary classification for important or not important

# Function to select important sentences
def extract_summary(article):
    sentences = article.split(". ")
    encoded_sentences = tokenizer(sentences, padding=True, truncation=True, return_tensors="pt")
    with torch.no_grad():
        logits = model(**encoded_sentences).logits
    important_sentences = [sentences[i] for i in range(len(sentences)) if logits[i][1] > 0.5]
    return " ".join(important_sentences)

# Extract summaries for the test set
test_samples = dataset["test"].select(range(10))  # Use a small subset for quick results
extracted_summaries = [extract_summary(sample["article"]) for sample in test_samples]

# Print the first extracted summary
print("Original Article:", test_samples[0]["article"])
print("Extracted Summary:", extracted_summaries[0])


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Original Article: (CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territories. The formal accession was marked with a ceremony at The Hague, in the Netherlands, where the court is based. The Palestinians signed the ICC's founding Rome Statute in January, when they also accepted its jurisdiction over alleged crimes committed "in the occupied Palestinian territory, including East Jerusalem, since June 13, 2014." Later that month, the ICC opened a preliminary examination into the situation in Palestinian territories, paving the way for possible war crimes investigations against Israelis. As members of the court, Palestinians may be subject to counter-charges as well. Israel and the United States, neither of which is an ICC member, opposed the Palestinians' efforts to join the body. But Palestinian Foreign Minister Riad al-Malki, speaking at Wednes

In [ ]:
# Reference summaries
reference_summaries = [sample["highlights"] for sample in test_samples]

# Calculate Rouge scores for BERT
bert_rouge_scores = calculate_rouge_scores(extracted_summaries, reference_summaries)
print("BERT Rouge Scores:", bert_rouge_scores)


BERT Rouge Scores: {'rouge1': 0.0, 'rouge2': 0.0, 'rougeL': 0.0}
